In [ ]:
# -*- coding: utf-8 -*-
"""
BrainVital8 DCA - Standardized Net Benefit
============================================
Y-axis: Standardized Net Benefit = NB / prevalence (0-1 range)
X-axis: High Risk Threshold (0-0.4)
Style: Clean, no grid, bold lines
"""

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from lifelines import CoxPHFitter
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.unicode_minus": False,
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
})


# ========================= Configuration =========================
DATA_FILE = r"./data/SEM_Analysis/All_cohort_final_BrainVital8.csv"
OUTPUT_DIR = r"./results/SEM_Analysis/DCA_results"

COHORT_CONFIG = {
    "ukb":    {"time_point": 15, "color": "#8B4513", "label": "UKB"},
    "charls": {"time_point": 7,  "color": "#3498db", "label": "CHARLS"},
    "elsa":   {"time_point": 7,  "color": "#e74c3c", "label": "ELSA"},
    "hrs":    {"time_point": 10, "color": "#16a085", "label": "HRS"},
    "share":  {"time_point": 10, "color": "#00BCD4", "label": "SHARE"},
    "klosa":  {"time_point": 7,  "color": "#9b59b6", "label": "KLoSA"},
}

COVARIATES = ["baseline_age", "gender", "bmi", "drinking", "hypertension", "scores"]
N_FOLDS = 5
RANDOM_SEED = 42
THRESHOLD_RANGE = np.arange(0.001, 0.41, 0.001)
KEY_THRESHOLDS = [0.01, 0.02, 0.03, 0.05]


# ========================= Data Loading =========================

def load_and_split(filepath):
    """Load data, rename columns, split by cohort."""
    df = pd.read_csv(filepath)
    df = df.rename(columns={
        "age": "baseline_age", "sex": "gender",
        "drink": "drinking", "BrainVital8": "scores",
    })
    for col in COVARIATES + ["time", "status"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[df["time"] > 0]
    df["cohort"] = df["cohort"].str.lower().str.strip()

    cohorts = {}
    for name in sorted(df["cohort"].unique()):
        sub = df[df["cohort"] == name][COVARIATES + ["time", "status"]].dropna().reset_index(drop=True)
        cohorts[name] = sub
    return cohorts


# ========================= Prediction =========================

def get_oof_predictions(df, covariates, target_year, k=5):
    """Get out-of-fold risk predictions using Cox regression with K-fold CV."""
    pred = np.zeros(len(df))
    kf = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
    for tr_idx, va_idx in kf.split(df):
        df_tr = df.iloc[tr_idx].reset_index(drop=True)
        df_va = df.iloc[va_idx].reset_index(drop=True)
        try:
            cph = CoxPHFitter()
            cph.fit(df_tr[covariates + ["time", "status"]],
                    duration_col="time", event_col="status")
            surv = cph.predict_survival_function(df_va, times=[target_year])
            pred[va_idx] = 1 - surv.values.flatten()
        except Exception:
            pred[va_idx] = df["status"].mean()
    return np.clip(pred, 0, 1)


# ========================= DCA Computation =========================

def compute_standardized_nb(pred_risk, time, status, target_year, thresholds):
    """Compute standardized Net Benefit (NB / prevalence) across thresholds."""
    N = len(pred_risk)
    event_at_t = ((time <= target_year) & (status == 1)).astype(int)
    prevalence = event_at_t.mean()

    results = []
    for pt in thresholds:
        test_positive = (pred_risk >= pt).astype(int)
        tp = np.sum((test_positive == 1) & (event_at_t == 1))
        fp = np.sum((test_positive == 1) & (event_at_t == 0))

        # Raw Net Benefit
        nb_model = tp / N - fp / N * pt / (1 - pt)
        nb_treat_all = prevalence - (1 - prevalence) * pt / (1 - pt)

        # Standardize by dividing by prevalence
        if prevalence > 0:
            snb_model = nb_model / prevalence
            snb_treat_all = nb_treat_all / prevalence
        else:
            snb_model = 0
            snb_treat_all = 0

        snb_model = max(snb_model, -0.1)
        snb_treat_all = max(snb_treat_all, -0.1)

        # Net reduction in unnecessary interventions
        if 0 < pt < 1:
            net_reduction = (nb_model - nb_treat_all) / (pt / (1 - pt))
        else:
            net_reduction = 0

        results.append({
            "threshold": pt,
            "snb_model": snb_model,
            "snb_treat_all": snb_treat_all,
            "nb_model": nb_model,
            "nb_treat_all": nb_treat_all,
            "net_reduction_per_100": net_reduction * 100,
        })

    return pd.DataFrame(results), prevalence


def compute_sens_spec(pred_risk, time, status, target_year, pt):
    """Compute sensitivity and specificity at a given threshold."""
    event_at_t = ((time <= target_year) & (status == 1)).astype(int)
    test_pos = (pred_risk >= pt).astype(int)
    tp = np.sum((test_pos == 1) & (event_at_t == 1))
    fn = np.sum((test_pos == 0) & (event_at_t == 1))
    fp = np.sum((test_pos == 1) & (event_at_t == 0))
    tn = np.sum((test_pos == 0) & (event_at_t == 0))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    screened = int(test_pos.sum())
    return sens, spec, tp, fn, screened


def smooth_curve(x, y, frac=0.1):
    """LOWESS smoothing for DCA curves."""
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        smoothed = lowess(y, x, frac=frac, return_sorted=True)
        return smoothed[:, 0], smoothed[:, 1]
    except ImportError:
        window = max(5, len(y) // 20)
        y_smooth = pd.Series(y).rolling(window=window, center=True, min_periods=1).mean().values
        return x, y_smooth


# ========================= Plotting =========================

def draw_dca_subplot(ax, dca_df, color, title):
    """Draw a single DCA subplot with LOWESS-smoothed model curve."""
    # Treat All (grey)
    ax.plot(dca_df["threshold"], dca_df["snb_treat_all"],
            "-", color="#999999", linewidth=1.5, label="All")

    # Treat None (black, y=0)
    ax.axhline(y=0, color="black", linewidth=1.2, label="None")

    # Model curve (LOWESS smoothed)
    x_smooth, y_smooth = smooth_curve(
        dca_df["threshold"].values, dca_df["snb_model"].values, frac=0.15)
    ax.plot(x_smooth, y_smooth,
            "-", color=color, linewidth=2.5, label="BrainVital8")

    ax.set_xlim([0, 0.4])
    ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel("High Risk Threshold")
    ax.set_ylabel("")
    ax.set_title(title, fontsize=14, fontweight="bold", pad=10)
    ax.legend(loc="upper right", frameon=False, fontsize=9)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4])
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])


def figure_dca_main(all_dca, output_path):
    """Main figure: 2x3 layout for 6 cohorts."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    cohort_names = list(all_dca.keys())
    for idx, name in enumerate(cohort_names):
        if idx >= 6:
            break
        cfg = COHORT_CONFIG[name]
        draw_dca_subplot(axes[idx], all_dca[name]["dca_df"],
                         cfg["color"], cfg["label"])
        if idx % 3 == 0:
            axes[idx].set_ylabel("Standardized Net Benefit", fontsize=12)

    # Hide unused panels
    for idx in range(len(cohort_names), 6):
        axes[idx].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(output_path + ".pdf", bbox_inches="tight")
    plt.close()
    print(f"  DCA main figure saved: {output_path}.png/.pdf")


def figure_dca_individual(all_dca, output_dir):
    """Individual large figure for each cohort."""
    for name, dca_info in all_dca.items():
        cfg = COHORT_CONFIG[name]
        fig, ax = plt.subplots(figsize=(6, 5))
        draw_dca_subplot(ax, dca_info["dca_df"], cfg["color"], cfg["label"])
        ax.set_ylabel("Standardized Net Benefit", fontsize=12)
        plt.tight_layout()
        p = os.path.join(output_dir, f"DCA_individual_{cfg['label']}")
        plt.savefig(p + ".png", dpi=300, bbox_inches="tight")
        plt.savefig(p + ".pdf", bbox_inches="tight")
        plt.close()
        print(f"  Individual figure saved: {p}.png/.pdf")


# ========================= Main =========================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 80)
    print("  BrainVital8 DCA - Standardized Net Benefit")
    print("=" * 80)

    cohorts = load_and_split(DATA_FILE)
    all_dca = {}
    table_rows = []

    for name in COHORT_CONFIG:
        if name not in cohorts:
            continue

        cfg = COHORT_CONFIG[name]
        df = cohorts[name]
        year = cfg["time_point"]

        print(f"\nProcessing: {cfg['label']} ({year}-year), N={len(df):,}")

        if len(df) < 100 or df["status"].sum() < 20:
            print("  Warning: insufficient samples, skipping")
            continue

        # Out-of-fold predictions
        pred = get_oof_predictions(df, COVARIATES, year, k=N_FOLDS)

        # Standardized DCA
        dca_df, prevalence = compute_standardized_nb(
            pred, df["time"].values, df["status"].values, year, THRESHOLD_RANGE)

        all_dca[name] = {"dca_df": dca_df, "year": year, "pred": pred,
                         "df": df, "prevalence": prevalence}

        print(f"  Event rate: {prevalence * 100:.2f}%")

        # Metrics at key thresholds
        for pt in KEY_THRESHOLDS:
            row = dca_df.iloc[np.argmin(np.abs(dca_df["threshold"] - pt))]
            sens, spec, tp, fn, screened = compute_sens_spec(
                pred, df["time"].values, df["status"].values, year, pt)
            print(f"    pt={pt * 100:.0f}%: NB={row['nb_model']:.5f}, "
                  f"Sens={sens * 100:.1f}%, Spec={spec * 100:.1f}%, "
                  f"NR={row['net_reduction_per_100']:.0f}/100")

            table_rows.append({
                "Cohort": cfg["label"],
                "Horizon (y)": year,
                "Event Rate (%)": round(prevalence * 100, 2),
                "Threshold (%)": int(pt * 100),
                "Net Benefit": round(row["nb_model"], 5),
                "Sensitivity (%)": round(sens * 100, 1),
                "Specificity (%)": round(spec * 100, 1),
                "Screened": screened,
                "Net Reduction": int(round(row["net_reduction_per_100"])),
                "Model Useful": row["nb_model"] > 0 and row["nb_model"] > row["nb_treat_all"],
            })

    # --- Figures ---
    print("\nGenerating figures...")
    figure_dca_main(all_dca, os.path.join(OUTPUT_DIR, "Figure_DCA_standardized"))
    figure_dca_individual(all_dca, OUTPUT_DIR)

    # --- Summary table ---
    table_df = pd.DataFrame(table_rows)
    table_df.to_csv(os.path.join(OUTPUT_DIR, "Table_DCA_standardized.csv"),
                    index=False, encoding="utf-8-sig")

    # --- Print summary ---
    print("\n" + "=" * 120)
    print("  DCA Summary Table")
    print("=" * 120)

    print(f"\n  {'Cohort':<8} {'Year':>4} {'Rate':>6} {'Thresh':>6} "
          f"{'NB':>10} {'Sens':>8} {'Spec':>8} {'Screen':>8} {'NR':>6} {'Useful':>6}")
    print(f"  {'-' * 80}")
    for _, r in table_df.iterrows():
        useful_str = "Yes" if r["Model Useful"] else "No"
        print(f"  {r['Cohort']:<8} {r['Horizon (y)']:>4} {r['Event Rate (%)']:>5.1f}% "
              f"{r['Threshold (%)']:>5}% "
              f"{r['Net Benefit']:>10.5f} {r['Sensitivity (%)']:>7.1f}% "
              f"{r['Specificity (%)']:>7.1f}% {r['Screened']:>8,} {r['Net Reduction']:>5}/100 "
              f"{useful_str:>6}")

    # Compact version (2% and 5% thresholds only)
    print(f"\n\n  Compact View (2% and 5% thresholds)")
    print(f"  {'Cohort':<8} {'Year':>4} {'Rate':>6}  "
          f"{'Sens@2%':>8} {'NR@2%':>8}  {'Sens@5%':>8} {'NR@5%':>8}")
    print(f"  {'-' * 65}")
    for name in COHORT_CONFIG:
        cfg = COHORT_CONFIG[name]
        sub = table_df[table_df["Cohort"] == cfg["label"]]
        if sub.empty:
            continue
        r2 = sub[sub["Threshold (%)"] == 2]
        r5 = sub[sub["Threshold (%)"] == 5]
        if r2.empty or r5.empty:
            continue
        r2, r5 = r2.iloc[0], r5.iloc[0]
        print(f"  {cfg['label']:<8} {r2['Horizon (y)']:>4} {r2['Event Rate (%)']:>5.1f}%  "
              f"{r2['Sensitivity (%)']:>7.1f}% {r2['Net Reduction']:>5}/100  "
              f"{r5['Sensitivity (%)']:>7.1f}% {r5['Net Reduction']:>5}/100")

    print(f"\nAll outputs saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()